КТ3, Лазарева А.К., 3П3.24

Задание 1

In [1]:
%%writefile test_task1.py
import pytest


def is_even(number):
    if not isinstance(number, int):
        raise TypeError
    return number % 2 == 0


@pytest.mark.parametrize("number, expected", [
    (2, True),
    (7, False),
    (0, True),
    (-4, True),
    (-3, False),
])
def test_is_even(number, expected):
    assert is_even(number) == expected


@pytest.mark.parametrize("number", ["5", 2.5, None])
def test_is_even_error(number):
    with pytest.raises(TypeError):
        is_even(number)

Writing test_task1.py


In [2]:
!pytest test_task1.py -q

........                                                                 [100%]
8 passed in 0.03s


Задание 2

In [3]:
%%writefile test_task2.py
import pytest


def calculate_area(length, width):
    if length <= 0 or width <= 0:
        raise ValueError
    return length * width


@pytest.mark.parametrize("length, width, expected", [
    (2, 3, 6),
    (5, 5, 25),
    (10, 1, 10),
    (4, 2.5, 10),
    (1.5, 2, 3),
])
def test_area(length, width, expected):
    assert calculate_area(length, width) == expected


@pytest.mark.parametrize("length, width", [(0, 5), (-2, 3), (4, -1)])
def test_area_error(length, width):
    with pytest.raises(ValueError):
        calculate_area(length, width)

Writing test_task2.py


In [4]:
!pytest test_task2.py -q

........                                                                 [100%]
8 passed in 0.03s


Задание 3

In [5]:
%%writefile test_task3.py
import pytest


def classify_triangle(a, b, c):
    if a <= 0 or b <= 0 or c <= 0:
        raise ValueError
    if a + b <= c or a + c <= b or b + c <= a:
        raise ValueError
    if a == b == c:
        return "равносторонний"
    if a == b or a == c or b == c:
        return "равнобедренный"
    return "разносторонний"


@pytest.mark.parametrize("a, b, c", [(3, 3, 3), (10, 10, 10)])
def test_equilateral(a, b, c):
    assert classify_triangle(a, b, c) == "равносторонний"


@pytest.mark.parametrize("a, b, c", [(5, 5, 3), (5, 3, 5), (3, 5, 5)])
def test_isosceles(a, b, c):
    assert classify_triangle(a, b, c) == "равнобедренный"


@pytest.mark.parametrize("a, b, c", [(3, 4, 5), (6, 7, 8)])
def test_scalene(a, b, c):
    assert classify_triangle(a, b, c) == "разносторонний"


@pytest.mark.parametrize("a, b, c", [(1, 2, 3), (1, 1, 10), (0, 4, 5), (-3, 4, 5)])
def test_not_triangle(a, b, c):
    with pytest.raises(ValueError):
        classify_triangle(a, b, c)

Writing test_task3.py


In [6]:
!pytest test_task3.py -q

...........                                                              [100%]
11 passed in 0.03s


Задание 4 (доработка КТ2)

In [7]:
%%writefile test_task4.py
import sqlite3

import pytest


def string_length(text):
    if not isinstance(text, str):
        raise TypeError
    return len(text)


def save_to_file(text, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def add_user(conn, name, age):
    conn.execute("INSERT INTO users (name, age) VALUES (?, ?)", (name, age))
    conn.commit()


@pytest.fixture(params=["", "привет", "   ", "первая\nвторая"], ids=["empty", "word", "spaces", "multiline"])
def text(request):
    return request.param


@pytest.fixture
def db(tmp_path):
    conn = sqlite3.connect(tmp_path / "test.db")
    conn.execute("CREATE TABLE users (name TEXT NOT NULL, age INTEGER NOT NULL CHECK (age >= 0))")
    yield conn
    conn.execute("DELETE FROM users")
    conn.commit()
    conn.close()


@pytest.mark.parametrize("value, expected", [
    ("", 0),
    ("hello", 5),
    ("   ", 3),
    ("a\nb\nc", 5),
], ids=["empty", "word", "spaces", "multiline"])
def test_string_length(value, expected):
    assert string_length(value) == expected


def test_save_to_file(tmp_path, text):
    path = tmp_path / "result.txt"
    save_to_file(text, path)
    assert path.read_text(encoding="utf-8") == text


@pytest.mark.parametrize("name, age", [("Alina", 19), ("Ivan", 0)])
def test_add_user(db, name, age):
    add_user(db, name, age)
    assert db.execute("SELECT name, age FROM users").fetchall() == [(name, age)]


@pytest.mark.parametrize("name, age", [("Oleg", -1), (None, 20)])
def test_add_user_error(db, name, age):
    with pytest.raises(sqlite3.IntegrityError):
        add_user(db, name, age)

Writing test_task4.py


In [8]:
!pytest test_task4.py -q

............                                                             [100%]
12 passed in 0.22s
